# Forcasting Multiple Articles

In [ ]:
# Install Prophet if needed
!pip install prophet

import pandas as pd
import numpy as np
from prophet import Prophet
import matplotlib.pyplot as plt
from google.colab import files
import io
import warnings

# Suppress annoying prophet warnings
warnings.filterwarnings("ignore")

# 1. Upload Data
print("📂 Please click 'Choose Files' to upload your BigQuery Master Extract:")
uploaded = files.upload()
filename = next(iter(uploaded))
print(f"⚙️ Processing file: {filename}")

# Read the file
df_main = pd.read_csv(io.BytesIO(uploaded[filename]))
df_main['WeekStartDate'] = pd.to_datetime(df_main['WeekStartDate'])

unique_articles = df_main['article'].unique()
print(f"📦 Found {len(unique_articles)} unique articles to forecast.\n")
all_forecasts = []

# =========================================================
# 🔄 START BATCH LOOP: Process each article one by one
# =========================================================
for article_id in unique_articles:
    try:
        print(f"--------------------------------------------------")
        print(f"🚀 Processing Article: {article_id}")

        df = df_main[df_main['article'] == article_id].copy()
        df = df.sort_values(by='WeekStartDate')

        full_history = df[df['ActDmd'].notnull()].copy()
        total_history_weeks = len(full_history)

        if total_history_weeks < 4:
            print(f"⚠️ Skipping {article_id}: Not enough historical data.")
            continue

        # Handle ramp-up period for new lines
        if 12 < total_history_weeks < 104:
            mature_history = full_history.iloc[12:].copy()
        else:
            mature_history = full_history.copy()

        # ---------------------------------------------------------
        # 🛠️ Data Cleaning & Drop-off Fix (UPDATED for NaNs)
        # ---------------------------------------------------------
        # Calculate recent store count robustly to prevent NaN errors
        valid_store_counts = mature_history['StoreCount'].replace(0, np.nan).dropna()
        recent_store_count = valid_store_counts.iloc[-4:].mean() if not valid_store_counts.empty else 1

        df['StoreCount'] = df['StoreCount'].replace(0, np.nan).fillna(recent_store_count)
        df['price'] = df['price'].replace(0, np.nan).fillna(method='ffill').fillna(method='bfill')

        df['PromStoreCount'] = df['PromStoreCount'].fillna(0)
        df['discount_pcnt'] = df['discount_pcnt'].fillna(0)

        future_dates = df[df['ActDmd'].isnull()].copy()
        if len(future_dates) == 0:
            continue

        # ---------------------------------------------------------
        # 🛠️ Format for Prophet (Streamlined Features Only!)
        # ---------------------------------------------------------
        # CRITICAL FIX: Extract training data from the CLEANED df using mature_history index
        prophet_train = df.loc[mature_history.index, ['WeekStartDate', 'ActDmd', 'discount_pcnt', 'StoreCount', 'PromStoreCount']].copy()
        prophet_train.columns = ['ds', 'y', 'discount_pcnt', 'StoreCount', 'PromStoreCount']

        valid_weeks = len(mature_history)
        if valid_weeks < 52:
            y_seas = False
            trend_growth = 'flat'
        else:
            y_seas = True
            trend_growth = 'linear'

        # Initialize Additive Model
        model = Prophet(
            growth=trend_growth,
            yearly_seasonality=y_seas,
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode='additive'
        )

        # Add Regressors: Only the pure, numeric drivers of elasticity!
        model.add_regressor('discount_pcnt')
        model.add_regressor('StoreCount')
        model.add_regressor('PromStoreCount')

        # Train Model
        model.fit(prophet_train)

        # Predict Future
        prophet_future = future_dates[['WeekStartDate', 'discount_pcnt', 'StoreCount', 'PromStoreCount']].copy()
        prophet_future.columns = ['ds', 'discount_pcnt', 'StoreCount', 'PromStoreCount']

        forecast = model.predict(prophet_future)
        forecast['yhat'] = forecast['yhat'].apply(lambda x: max(0, x))

        future_dates['Generated_Forecast'] = forecast['yhat'].values
        output_cols = ['WeekStartDate', 'article', 'price', 'discount_pcnt', 'StoreCount', 'PromStoreCount', 'Generated_Forecast']
        all_forecasts.append(future_dates[output_cols])

        # ---------------------------------------------------------
        # 8. Visualise with Dual Y-Axis
        # ---------------------------------------------------------
        fig, ax1 = plt.subplots(figsize=(10, 4))

        # --- Axis 1: Demand (Left Side) ---
        if 12 < total_history_weeks < 104:
            skipped = full_history.iloc[:12]
            ax1.plot(skipped['WeekStartDate'], skipped['ActDmd'], label='Skipped Ramp-Up', color='lightgrey', linestyle='--')

        ax1.plot(df.loc[mature_history.index, 'WeekStartDate'], df.loc[mature_history.index, 'ActDmd'], label='Training Data', color='blue')
        ax1.plot(future_dates['WeekStartDate'], future_dates['Generated_Forecast'], label=f"AI Forecast", color='orange')

        ax1.set_xlabel('Date')
        ax1.set_ylabel('Demand Volume', color='blue')
        ax1.tick_params(axis='y', labelcolor='blue')
        ax1.grid(True, alpha=0.3)

        # --- Axis 2: Price (Right Side) ---
        ax2 = ax1.twinx()  # Creates second axis sharing X-axis

        # Plot price across entire timeline
        ax2.plot(df['WeekStartDate'], df['price'], label='Price ($)', color='green', linestyle=':', linewidth=2, alpha=0.7)
        ax2.set_ylabel('Price ($)', color='green')
        ax2.tick_params(axis='y', labelcolor='green')

        # Combine Legends and Title
        lines_1, labels_1 = ax1.get_legend_handles_labels()
        lines_2, labels_2 = ax2.get_legend_handles_labels()
        ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

        plt.title(f"Demand Forecast & Price | Article {article_id}")
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"❌ ERROR processing article {article_id}: {e}")
        continue

# =========================================================
# 🏁 END BATCH LOOP: Stitch and Download
# =========================================================
print("--------------------------------------------------")
if len(all_forecasts) > 0:
    master_output = pd.concat(all_forecasts, ignore_index=True)
    output_filename = f"Master_Forecast_Batch_{filename}"
    master_output.to_csv(output_filename, index=False)
    files.download(output_filename)
    print(f"✅ ALL DONE! Downloaded: {output_filename}")
else:
    print("⚠️ No forecasts were generated. Please check your data.")

📂 Please click 'Choose Files' to upload your BigQuery Master Extract:
